# Week 9 Lab: register and log in

Last week you fetched data. This week your project gets **users**: people can register and log in.
Their passwords are never stored. Only a **hash** is stored, made with bcrypt.

Today you add to your `CST1510_CW2` folder:

```
CST1510_CW2/
  auth.py             register and log in from the terminal
  DATA/users.txt      created when someone registers (stays on your computer)
```

`fetch_data.py` and `DATA/raw.csv` from Week 8 stay exactly as they are.
Ideas to revisit: topics `01` (why hash), `02` (bcrypt) and `03` (register and login) in this week's folder.

In [2]:
# Run this cell first, every time you open the notebook.
import sys
from pathlib import Path

# Your project lives in the CST1510_CW2 folder. This notebook looks for it in the usual places.
# If it cannot find it, paste the full path between the quotes, for example "C:/Users/sam/Documents/CST1510_CW2"
MY_FOLDER = ""


def find_project():
    if MY_FOLDER:
        return Path(MY_FOLDER)
    here = Path.cwd()
    for folder in [here, *here.parents]:
        if folder.name == "CST1510_CW2":
            return folder
    for base in [Path.home(), Path.home() / "Documents", Path.home() / "Desktop", Path.home() / "OneDrive" / "Documents"]:
        if (base / "CST1510_CW2").exists():
            return base / "CST1510_CW2"
    return here


PROJECT = find_project()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Project folder:", PROJECT)
if PROJECT.name != "CST1510_CW2":
    print("\nI could not find your CST1510_CW2 folder. Paste its full path into MY_FOLDER above, then run this cell again.")


def show_checks(checks):
    """Print PASS or FAIL for each (passed, description) pair."""
    for passed, description in checks:
        print("PASS" if passed else "FAIL", "-", description)
    print("\nAll checks passed." if all(p for p, _ in checks) else "\nFix the lines marked FAIL, save, and run again.")


show_checks([
    (PROJECT.name == "CST1510_CW2",                 f"working in the CST1510_CW2 folder (this is {PROJECT.name})"),
    ((PROJECT / "fetch_data.py").exists(),          "fetch_data.py from Week 8 is here"),
    ((PROJECT / "DATA" / "raw.csv").exists(),       "DATA/raw.csv from Week 8 is here"),
])

Project folder: /content

I could not find your CST1510_CW2 folder. Paste its full path into MY_FOLDER above, then run this cell again.
FAIL - working in the CST1510_CW2 folder (this is content)
FAIL - fetch_data.py from Week 8 is here
FAIL - DATA/raw.csv from Week 8 is here

Fix the lines marked FAIL, save, and run again.


## Step 1: Hash a password

A website must never keep your real password. It keeps a **hash**: a scrambled version that cannot be turned back
into the password. bcrypt also adds a random **salt**, so the same password gives a different hash every time.

bcrypt works with **bytes**, so every password goes through `.encode("utf-8")` first.

**Try it:**

In [3]:
%pip install bcrypt

In [4]:
import bcrypt

password = "Magic1234"
byte_password = password.encode("utf-8")
salt = bcrypt.gensalt()
hashed = bcrypt.hashpw(byte_password, salt)

print("Bytes: ", byte_password)
print("Hash:  ", hashed)
print("As text:", hashed.decode("utf-8"))

Bytes:  b'Magic1234'
Hash:   b'$2b$12$bNBnB/IhKHEuJ.vGGDLdj.3cAjYvDCv9v0x3q6.uP9dYnJUQUTCne'
As text: $2b$12$bNBnB/IhKHEuJ.vGGDLdj.3cAjYvDCv9v0x3q6.uP9dYnJUQUTCne


**Exercise.** Run the cell three times. Is the hash the same each time? Why not? (Hint: the salt.)
Then change the password and run again.

## Step 2: Check a password

To log in you do not "unscramble" the hash. You check the typed password against it with `bcrypt.checkpw(password, hash)`.
Both must be bytes, and the order matters: **password first, hash second**.

**Try it:**

In [5]:
print(bcrypt.checkpw("Magic1234".encode("utf-8"), hashed))      # right password
print(bcrypt.checkpw("Wrong1234".encode("utf-8"), hashed))      # wrong password

True
False


**Exercise.** Swap the two arguments in the first line and run it. Read the error. Then put them back.

## Step 3: Two helper functions

You will use hashing and checking in many places, so wrap them in two small functions.
The stored hash will live in a text file, so the functions take and return **text** (strings), not bytes.

**Try it.** Complete the two functions (the steps are in the comments), then run the cell:

In [6]:
def generate_hash(password):
    """Turn a password into a bcrypt hash, returned as text."""
    byte_password = password.encode("utf-8")
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(byte_password, salt)
    return hashed.decode("utf-8")           # back to text so it can go in a file


def is_valid_hash(password, hashed):
    """True if the password matches the hash (both given as text)."""
    # 1. encode the password to bytes   2. encode the hash to bytes   3. return bcrypt.checkpw(...)
    return bcrypt.checkpw(password.encode("utf-8"), hashed.encode("utf-8"))


h1 = generate_hash("Magic1234")
h2 = generate_hash("Magic1234")
print("Same hash twice?", h1 == h2)                           # False: different salts
print("Right password:", is_valid_hash("Magic1234", h1))      # True
print("Wrong password:", is_valid_hash("Wrong1234", h1))      # False

Same hash twice? False
Right password: True
Wrong password: False


## Step 4: Keep users in a text file

Each user is one line: `username,hash`. A bcrypt hash never contains a comma, so the comma is a safe separator.

| Job | Code |
|---|---|
| add a line to the end of the file | `open(path, "a")` and `file.write(...)` |
| read every line | `open(path, "r")` and `for line in file:` |
| split a line into two parts | `line.strip().split(",", 1)` |

Two cases to handle: the file does not exist yet (the very first run), and a username that is already taken.

**Try it.** This uses a temporary file, so it will not touch your project:

In [7]:
import tempfile

practice = Path(tempfile.mkdtemp()) / "practice_users.txt"
print("File exists yet?", practice.exists())

with open(practice, "a") as file:                       # "a" = append: creates the file if needed
    file.write(f"sam,{generate_hash('Magic1234')}\n")

with open(practice, "r") as file:
    for line in file:
        name, stored_hash = line.strip().split(",", 1)
        print(name, "->", stored_hash[:20] + "...")

File exists yet? False
sam -> $2b$12$YjEClFUwtEl5d...


**Exercise.** Add a second user called `alex` the same way, then read the file back and print only the usernames.

## Step 5: Build `auth.py`

Now put it all together. Notice that `register_user` and `login_user` return **two things**: `True` or `False`,
and a message. In Week 11 the Streamlit page will show that message on screen, so keep this shape.

| Function | Job |
|---|---|
| `generate_hash`, `is_valid_hash` | Step 3 |
| `user_exists(username)` | is the name already in the file? |
| `register_user(username, password)` | checks the rules, then adds a line |
| `login_user(username, password)` | finds the line, then checks the password |
| `main()` | a small menu so you can try it |

Note the login gives the **same message** for an unknown user and a wrong password. That is on purpose: it does not tell
an attacker which usernames exist.

**Build.** Create `auth.py` in your `CST1510_CW2` folder and paste this in:

```python
"""
auth.py - Week 9: register and log in from the terminal.

Passwords are hashed with bcrypt. Users are kept in DATA/users.txt, one per line: username,hash

Run it:  python auth.py
"""

from pathlib import Path

import bcrypt

USERS_FILE = Path(__file__).parent / "DATA" / "users.txt"


def generate_hash(password):
    """Turn a password into a bcrypt hash (returned as text)."""
    byte_password = password.encode("utf-8")
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(byte_password, salt)
    return hashed.decode("utf-8")


def is_valid_hash(password, hashed):
    """True if the password matches the stored hash."""
    return bcrypt.checkpw(password.encode("utf-8"), hashed.encode("utf-8"))


def user_exists(username, path=USERS_FILE):
    """True if this username is already in the file."""
    if not path.exists():
        return False
    with open(path, "r") as file:
        for line in file:
            stored_name, _ = line.strip().split(",", 1)
            if stored_name.lower() == username.lower():
                return True
    return False


def register_user(username, password, path=USERS_FILE):
    """Add a new user. Returns (True or False, a message)."""
    username = username.strip()
    if username == "" or "," in username:
        return False, "Username cannot be empty or contain a comma."
    if len(password) < 8:
        return False, "Password must be at least 8 characters."
    if user_exists(username, path):
        return False, "That username is already taken."

    path.parent.mkdir(exist_ok=True)
    with open(path, "a") as file:
        file.write(f"{username},{generate_hash(password)}\n")
    return True, "Registered. You can log in now."


def login_user(username, password, path=USERS_FILE):
    """Check a username and password. Returns (True or False, a message)."""
    if path.exists():
        with open(path, "r") as file:
            for line in file:
                stored_name, stored_hash = line.strip().split(",", 1)
                if stored_name.lower() == username.strip().lower():
                    if is_valid_hash(password, stored_hash):
                        return True, "Welcome back!"
                    break
    return False, "Wrong username or password."      # the same message for both problems, on purpose


def main():
    """A small menu to try it out."""
    while True:
        print("\n1. Register\n2. Log in\n3. Exit")
        choice = input("> ")
        if choice == "3":
            print("Goodbye!")
            break
        if choice in ("1", "2"):
            name = input("Username: ")
            password = input("Password: ")
            if choice == "1":
                ok, message = register_user(name, password)
            else:
                ok, message = login_user(name, password)
            print(("OK: " if ok else "NO: ") + message)


if __name__ == "__main__":
    main()
```

In [8]:
from pathlib import Path

import bcrypt

USERS_FILE = Path("DATA") / "users.txt"

def generate_hash(password):
    """Turn a password into a bcrypt hash (returned as text)."""
    byte_password = password.encode("utf-8")
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(byte_password, salt)
    return hashed.decode("utf-8")


def is_valid_hash(password, hashed):
    """True if the password matches the stored hash."""
    return bcrypt.checkpw(password.encode("utf-8"), hashed.encode("utf-8"))


def user_exists(username, path=USERS_FILE):
    """True if this username is already in the file."""
    if not path.exists():
        return False
    with open(path, "r") as file:
        for line in file:
            stored_name, _ = line.strip().split(",", 1)
            if stored_name.lower() == username.lower():
                return True
    return False


def register_user(username, password, path=USERS_FILE):
    """Add a new user. Returns (True or False, a message)."""
    username = username.strip()
    if username == "" or "," in username:
        return False, "Username cannot be empty or contain a comma."
    if len(password) < 8:
        return False, "Password must be at least 8 characters."
    if user_exists(username, path):
        return False, "That username is already taken."

    path.parent.mkdir(exist_ok=True)
    with open(path, "a") as file:
        file.write(f"{username},{generate_hash(password)}\n")
    return True, "Registered. You can log in now."


def login_user(username, password, path=USERS_FILE):
    """Check a username and password. Returns (True or False, a message)."""
    if path.exists():
        with open(path, "r") as file:
            for line in file:
                stored_name, stored_hash = line.strip().split(",", 1)
                if stored_name.lower() == username.strip().lower():
                    if is_valid_hash(password, stored_hash):
                        return True, "Welcome back!"
                    break
    return False, "Wrong username or password."      # the same message for both problems, on purpose


def main():
    """A small menu to try it out."""
    while True:
        print("\n1. Register\n2. Log in\n3. Exit")
        choice = input("> ")
        if choice == "3":
            print("Goodbye!")
            break
        if choice in ("1", "2"):
            name = input("Username: ")
            password = input("Password: ")
            if choice == "1":
                ok, message = register_user(name, password)
            else:
                ok, message = login_user(name, password)
            print(("OK: " if ok else "NO: ") + message)


if __name__ == "__main__":
    main()


1. Register
2. Log in
3. Exit
> 2
Username: ali
Password: ali764719
OK: Welcome back!

1. Register
2. Log in
3. Exit
> 3
Goodbye!


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile auth.py


## Step 6: Check it

This test uses a temporary file, so it will not put fake users in your project. Run it:

In [11]:
def show_checks(checks):
    for passed, message in checks:
        if passed:
            print("PASS:", message)
        else:
            print("FAIL:", message)

In [12]:
%%writefile auth.py

from pathlib import Path
import bcrypt

USERS_FILE = Path(__file__).parent / "DATA" / "users.txt"


def generate_hash(password):
    byte_password = password.encode("utf-8")
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(byte_password, salt)
    return hashed.decode("utf-8")


def is_valid_hash(password, hashed):
    return bcrypt.checkpw(
        password.encode("utf-8"),
        hashed.encode("utf-8")
    )


def user_exists(username, path=USERS_FILE):
    if not path.exists():
        return False

    with open(path, "r") as file:
        for line in file:
            stored_name, _ = line.strip().split(",", 1)
            if stored_name.lower() == username.lower():
                return True

    return False


def register_user(username, password, path=USERS_FILE):
    username = username.strip()

    if username == "" or "," in username:
        return False, "Username cannot be empty or contain a comma."

    if len(password) < 8:
        return False, "Password must be at least 8 characters."

    if user_exists(username, path):
        return False, "That username is already taken."

    path.parent.mkdir(exist_ok=True)

    with open(path, "a") as file:
        file.write(f"{username},{generate_hash(password)}\n")

    return True, "Registered. You can log in now."


def login_user(username, password, path=USERS_FILE):
    if path.exists():
        with open(path, "r") as file:
            for line in file:
                stored_name, stored_hash = line.strip().split(",", 1)

                if stored_name.lower() == username.strip().lower():
                    if is_valid_hash(password, stored_hash):
                        return True, "Welcome back!"
                    break

    return False, "Wrong username or password."


def main():
    while True:
        print("\n1. Register\n2. Log in\n3. Exit")
        choice = input("> ")

        if choice == "3":
            print("Goodbye!")
            break

        if choice in ("1", "2"):
            name = input("Username: ")
            password = input("Password: ")

            if choice == "1":
                ok, message = register_user(name, password)
            else:
                ok, message = login_user(name, password)

            print(("OK: " if ok else "NO: ") + message)


if __name__ == "__main__":
    main()

Overwriting auth.py


In [13]:
import importlib
import auth

importlib.reload(auth)

print(hasattr(auth, "login_user"))
print(hasattr(auth, "register_user"))

True
True


In [14]:
import importlib
import tempfile
from pathlib import Path
import auth

importlib.reload(auth)

def show_checks(checks):
    for passed, message in checks:
        print("PASS:" if passed else "FAIL:", message)

test_file = Path(tempfile.mkdtemp()) / "users.txt"

show_checks([
    (auth.login_user("nobody", "whatever1", test_file)[0] is False,
     "login before any user exists does not crash"),

    (auth.register_user("anna", "short", test_file)[0] is False,
     "a password under 8 characters is refused"),

    (auth.register_user("anna", "longenough1", test_file)[0] is True,
     "register works"),

    (auth.register_user("Anna", "longenough1", test_file)[0] is False,
     "a duplicate username is refused (even with a capital)"),

    (auth.login_user("anna", "longenough1", test_file)[0] is True,
     "the right password is accepted"),

    (auth.login_user("anna", "wrong-password", test_file)[0] is False,
     "the wrong password is refused"),

    ("longenough1" not in test_file.read_text(),
     "the plain password is not in the file"),
])

PASS: login before any user exists does not crash
PASS: a password under 8 characters is refused
PASS: register works
PASS: a duplicate username is refused (even with a capital)
PASS: the right password is accepted
PASS: the wrong password is refused
PASS: the plain password is not in the file


**Try it by hand.** In the VS Code terminal, inside `CST1510_CW2`:

```
python auth.py
```

(`python3` on Mac and Linux.) Register yourself, choose 3 to quit, run it again and log in with the right and the wrong password.
Then run the cell below to see what was saved.

In [15]:
users_file = PROJECT / "DATA" / "users.txt"
print(users_file.read_text() if users_file.exists() else "No users yet: run  python auth.py  and register someone.")

ali,$2b$12$/g.q.qPbVjd0pLVIZxXYJOCGkP2ofFUiIE92.SRTJ2gWEM5kBLpgW



Look at the line. Can you find your password in it? The hash starts with `$2b$12$`: the `12` is the bcrypt cost, which makes it slow on purpose.

## Step 7: Keep it private

`users.txt` holds hashes, and they should not go to GitHub. Your `.gitignore` from Week 8 already lists it. Check that it works:

In [17]:
import subprocess
result = subprocess.run(["git", "check-ignore", "-v", "DATA/users.txt"], cwd=PROJECT, capture_output=True, text=True)
print(result.stdout.strip() or "NOT ignored: add  DATA/users.txt  to your .gitignore")

NOT ignored: add  DATA/users.txt  to your .gitignore


Three rules for the rest of the module:
- never `print()` a password,
- `.encode("utf-8")` before every bcrypt call,
- `checkpw(password, hash)`: password first.

## Summary

```
CST1510_CW2/
  fetch_data.py
  auth.py
  DATA/raw.csv
  DATA/users.txt     (on your computer only)
  .gitignore
```

**Commit your work.** In the VS Code terminal, inside `CST1510_CW2`:

```
git add .
git commit -m "Week 9: register and log in with bcrypt"
git push
```

Then open `CST1510_CW2` on GitHub. You should see `auth.py` but **not** `users.txt`.

## Common errors

| You see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'bcrypt'` | bcrypt is not installed | `python -m pip install bcrypt` |
| `TypeError: Strings must be encoded before hashing` | a string was given to bcrypt | add `.encode("utf-8")` |
| `ValueError: Invalid salt` | `checkpw` arguments swapped | password first, hash second |
| `AttributeError: 'str' object has no attribute 'decode'` | decoded something that is already text | only decode the result of `hashpw` |
| `ValueError: not enough values to unpack` | a blank or broken line in `users.txt` | delete `DATA/users.txt` and register again |
| `ImportError` / old behaviour after editing `auth.py` | the notebook remembered the old version | run `importlib.reload(auth)` or restart the kernel |

Still stuck? Read the **last line** of the error message, then compare your file with `DEMO/project/auth.py`.